In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

PRICE_FILE = "1_data_.xlsx"
EARNINGS_FILE = "2_data_.xlsx"
RFM_FILE = "RFM-11-sector.csv"
INDEX_FILE = "index.csv"

START_DATE = "2023-03-18"
END_DATE = "2026-03-25"

TOP_N = 9
MCAP_MIN = 500_000
TURNOVER_MIN = 5_000_000_000

OUTPUT_RESULT = "backtest_result_daily.csv"
OUTPUT_HOLDINGS = "weekly_holdings.csv"
OUTPUT_METRICS = "performance_metrics.csv"
OUTPUT_FIGURE = "final_chart.png"


def clean_code(x):
    return str(x).strip() if pd.notna(x) else np.nan


def get_date_cols(df, exclude_cols):
    date_cols = []
    for col in df.columns:
        if col in exclude_cols:
            continue
        try:
            pd.to_datetime(col, errors="raise")
            date_cols.append(col)
        except Exception:
            continue
    return date_cols


def melt_fnguide(df, keyword, value_name, date_col="date"):
    subset = df[df["아이템명"].astype(str).str.contains(keyword, na=False)].copy()

    id_cols = [c for c in ["코드", "코드명", "아이템명"] if c in subset.columns]
    date_cols = get_date_cols(subset, id_cols)

    subset = subset[id_cols + date_cols].copy()
    subset["코드"] = subset["코드"].map(clean_code)

    out = subset.melt(
        id_vars=id_cols,
        value_vars=date_cols,
        var_name=date_col,
        value_name=value_name
    )

    out[date_col] = pd.to_datetime(out[date_col], errors="coerce")
    out[value_name] = pd.to_numeric(out[value_name], errors="coerce")
    out = out.dropna(subset=[date_col])

    return out


def load_index_proxy(file_name):
    raw = pd.read_csv(file_name, header=8)

    idx = raw[raw["아이템명"].astype(str).str.contains("지수", na=False)].copy()

    id_cols = [c for c in ["코드", "코드명", "아이템명"] if c in idx.columns]
    date_cols = get_date_cols(idx, id_cols)

    idx = idx[id_cols + date_cols].copy()

    idx_long = idx.melt(
        id_vars=id_cols,
        value_vars=date_cols,
        var_name="date",
        value_name="index_level"
    )

    idx_long["date"] = pd.to_datetime(idx_long["date"], errors="coerce")
    idx_long["index_level"] = pd.to_numeric(idx_long["index_level"], errors="coerce")
    idx_long = idx_long.dropna(subset=["date", "index_level"])

    idx_long = idx_long.groupby("date", as_index=False)["index_level"].mean()
    idx_long = idx_long.sort_values("date").reset_index(drop=True)

    idx_long["benchmark_ret"] = idx_long["index_level"].pct_change().fillna(0)
    idx_long["benchmark_nav"] = (1 + idx_long["benchmark_ret"]).cumprod()

    return idx_long


def calculate_metrics(ret):
    ret = ret.dropna().copy()
    nav = (1 + ret).cumprod()

    total_return = nav.iloc[-1] - 1
    annual_return = (1 + total_return) ** (252 / len(ret)) - 1
    volatility = ret.std() * np.sqrt(252)
    sharpe_ratio = annual_return / volatility if volatility != 0 else np.nan
    drawdown = nav / nav.cummax() - 1
    mdd = drawdown.min()

    return {
        "Total Return": total_return,
        "Annual Return": annual_return,
        "Volatility": volatility,
        "Sharpe Ratio": sharpe_ratio,
        "Maximum Drawdown": mdd
    }


price_raw = pd.read_excel(PRICE_FILE, header=8)
earn_raw = pd.read_excel(EARNINGS_FILE, header=8)
rfm = pd.read_csv(RFM_FILE)
index_df = load_index_proxy(INDEX_FILE)

rfm_codes = set(rfm["종목코드"].astype(str).str.strip())

price = melt_fnguide(price_raw, "수정주가", "price")
mcap = melt_fnguide(price_raw, "시가총액", "mcap")
turn = melt_fnguide(price_raw, "거래대금", "turnover")

df = price.merge(mcap[["코드", "date", "mcap"]], on=["코드", "date"], how="left")
df = df.merge(turn[["코드", "date", "turnover"]], on=["코드", "date"], how="left")

df["코드"] = df["코드"].map(clean_code)
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.dropna(subset=["코드", "date"])

df = df[df["코드"].isin(rfm_codes)].copy()
df = df[(df["date"] >= pd.to_datetime(START_DATE)) & (df["date"] <= pd.to_datetime(END_DATE))].copy()
df = df.sort_values(["코드", "date"]).reset_index(drop=True)

df["ret_1m"] = df.groupby("코드")["price"].pct_change(20)
df["ret_3m"] = df.groupby("코드")["price"].pct_change(60)
df["daily_ret"] = df.groupby("코드")["price"].pct_change()

earn = melt_fnguide(earn_raw, "영업이익", "op_income", "quarter")
earn["코드"] = earn["코드"].map(clean_code)
earn["quarter"] = pd.to_datetime(earn["quarter"], errors="coerce")
earn = earn.dropna(subset=["코드", "quarter"]).copy()
earn = earn.sort_values(["코드", "quarter"]).reset_index(drop=True)

earn["op_lag4"] = earn.groupby("코드")["op_income"].shift(4)
earn["op_yoy"] = (earn["op_income"] - earn["op_lag4"]) / earn["op_lag4"].abs()
earn["op_yoy"] = earn["op_yoy"].replace([np.inf, -np.inf], np.nan).clip(-5, 5)
earn = earn[["코드", "quarter", "op_yoy"]].drop_duplicates(subset=["코드", "quarter"], keep="last")

merged = []
for code, g in df.groupby("코드", sort=False):
    e = earn[earn["코드"] == code].sort_values("quarter").copy()
    g = g.sort_values("date").reset_index(drop=True)

    if e.empty:
        g["op_yoy"] = np.nan
    else:
        g = pd.merge_asof(
            g,
            e,
            left_on="date",
            right_on="quarter",
            direction="backward",
            allow_exact_matches=True
        )
    merged.append(g)

df = pd.concat(merged, ignore_index=True)
df = df.sort_values(["코드", "date"]).reset_index(drop=True)

df = df[
    (df["mcap"] >= MCAP_MIN) &
    (df["turnover"] >= TURNOVER_MIN)
].copy()

df["rank_1m"] = df.groupby("date")["ret_1m"].rank(ascending=False, method="min")
df["rank_3m"] = df.groupby("date")["ret_3m"].rank(ascending=False, method="min")
df["rank_op"] = df.groupby("date")["op_yoy"].rank(ascending=False, method="min")

df = df.dropna(subset=["rank_1m", "rank_3m", "rank_op"]).copy()
df["score"] = df["rank_1m"] + df["rank_3m"] + df["rank_op"]

weeks = df["date"].dt.to_period("W-FRI")
rebalance_dates = df.groupby(weeks)["date"].max().sort_values().tolist()

portfolio_records = []

for dt in rebalance_dates:
    snapshot = df[df["date"] == dt].copy()
    if snapshot.empty:
        continue

    selected = snapshot.nsmallest(TOP_N, "score").copy()
    selected["rebalance_date"] = dt
    portfolio_records.append(selected)

if len(portfolio_records) == 0:
    raise ValueError("No portfolio was formed. Try relaxing filter conditions.")

portfolio = pd.concat(portfolio_records, ignore_index=True)

portfolio[["rebalance_date", "코드", "코드명", "score"]].to_csv(
    OUTPUT_HOLDINGS, index=False, encoding="utf-8-sig"
)

strategy_ret_list = []
rebalance_list = sorted(portfolio["rebalance_date"].unique())

for i in range(len(rebalance_list)):
    start_dt = rebalance_list[i]
    end_dt = rebalance_list[i + 1] if i < len(rebalance_list) - 1 else df["date"].max()

    holdings = portfolio.loc[portfolio["rebalance_date"] == start_dt, "코드"].tolist()

    period = df[
        (df["date"] > start_dt) &
        (df["date"] <= end_dt) &
        (df["코드"].isin(holdings))
    ].copy()

    if period.empty:
        continue

    daily_port_ret = period.groupby("date")["daily_ret"].mean()
    strategy_ret_list.append(daily_port_ret)

strategy_ret = pd.concat(strategy_ret_list).sort_index().fillna(0)
strategy_nav = (1 + strategy_ret).cumprod()
strategy_nav = strategy_nav / strategy_nav.iloc[0]

result = pd.DataFrame({
    "date": strategy_nav.index,
    "strategy_ret": strategy_ret.values,
    "strategy_nav": strategy_nav.values
})

index_df = index_df[
    (index_df["date"] >= pd.to_datetime(START_DATE)) &
    (index_df["date"] <= pd.to_datetime(END_DATE))
].copy()

result = result.merge(
    index_df[["date", "benchmark_ret", "benchmark_nav"]],
    on="date",
    how="left"
)

result["benchmark_ret"] = result["benchmark_ret"].fillna(0)
result["benchmark_nav"] = (1 + result["benchmark_ret"]).cumprod()
result["benchmark_nav"] = result["benchmark_nav"] / result["benchmark_nav"].iloc[0]
result["drawdown"] = result["strategy_nav"] / result["strategy_nav"].cummax() - 1

result.to_csv(OUTPUT_RESULT, index=False, encoding="utf-8-sig")

metrics_df = pd.DataFrame({
    "Strategy": calculate_metrics(result["strategy_ret"]),
    "Benchmark": calculate_metrics(result["benchmark_ret"])
})
metrics_df.to_csv(OUTPUT_METRICS, encoding="utf-8-sig")

plt.figure(figsize=(14, 8))

ax1 = plt.subplot(2, 1, 1)
ax1.plot(result["date"], result["strategy_nav"], label="Rank Dual Momentum", linewidth=2)
ax1.plot(
    result["date"],
    result["benchmark_nav"],
    linestyle="--",
    linewidth=1.5,
    label="KOSPI200 sector TR proxy"
)
ax1.set_title("Rank-based Dual Momentum Backtest", fontsize=16)
ax1.set_ylabel("Cumulative Return (x)", fontsize=12)
ax1.legend(loc="upper left")
ax1.grid(alpha=0.25)
ax1.xaxis.set_major_locator(mdates.YearLocator())
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

ax2 = plt.subplot(2, 1, 2, sharex=ax1)
ax2.fill_between(result["date"], result["drawdown"] * 100, 0, alpha=0.25)
ax2.plot(result["date"], result["drawdown"] * 100, linewidth=1)
ax2.set_title("Drawdown", fontsize=14)
ax2.set_ylabel("Drawdown (%)", fontsize=12)
ax2.grid(alpha=0.25)
ax2.xaxis.set_major_locator(mdates.YearLocator())
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

plt.tight_layout()
plt.savefig(OUTPUT_FIGURE, dpi=300)
plt.show()

print("\n[Strategy Metrics]")
for k, v in calculate_metrics(result["strategy_ret"]).items():
    if pd.notna(v):
        if "Ratio" in k:
            print(f"{k}: {v:.3f}")
        else:
            print(f"{k}: {v:.2%}")
    else:
        print(f"{k}: NaN")

print("\n[Benchmark Metrics]")
for k, v in calculate_metrics(result["benchmark_ret"]).items():
    if pd.notna(v):
        if "Ratio" in k:
            print(f"{k}: {v:.3f}")
        else:
            print(f"{k}: {v:.2%}")
    else:
        print(f"{k}: NaN")